In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
file = "/content/drive/MyDrive/fitrain/data1.jsonl"


In [ ]:
/content/drive/MyDrive/fitrain/qwen-medical-lora

In [ ]:
import torch
torch.cuda.empty_cache()


In [ ]:
!pip install git+https://github.com/huggingface/transformers.git
!pip install --upgrade accelerate peft bitsandbytes datasets sentencepiece


  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-1j3ks3bf
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-1j3ks3bf
  Resolved https://github.com/huggingface/transformers.git to commit 549835e55228ea9158ce39e7b53a740bd8461594
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11464867 sha256=85311ac259b45c58de58bb89ce104c948d68db9a46b472187d920712438d043f
  Stored in directory: /tmp/pip-ephem-wheel-cache-5n7mr0ch/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model


In [ ]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# ===========================================
# 3) LOAD DATASET
# ===========================================
dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/fitrain/data1.jsonl"
)

def format_sample(example):
    prompt = f"""Instruction: {example['instruction']}

Patient Test Results:
{example['input']}

Answer:"""
    return {"text": prompt + " " + example["output"]}

dataset = dataset.map(format_sample)

# ===========================================
# 4) LOAD BASE MODEL (Qwen2.5-3B-Instruct)
# ===========================================
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False

# ===========================================
# 5) PREPARE LoRA CONFIG
# ===========================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# ===========================================
# 6) TOKENIZATION + LABELS
# ===========================================
def tokenize(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized = dataset.map(tokenize, batched=True)

# ===========================================
# 7) TRAINING ARGUMENTS
# ===========================================
training_args = TrainingArguments(
    output_dir="./qwen-medical-lora-3B",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    max_steps=500,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_steps=200,
    save_total_limit=3,
    optim="paged_adamw_32bit",
)

# ===========================================
# 8) ENABLE GRADIENT CHECKPOINTING
# ===========================================
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# ===========================================
# 9) TRAIN MODEL
# ===========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"]
)

trainer.train()

# ===========================================
# 10) SAVE LoRA MODEL
# ===========================================
model.save_pretrained("/content/drive/MyDrive/fitrain/qwen-medical-lora")
tokenizer.save_pretrained("/content/drive/MyDrive/fitrain/qwen-medical-lora")

print("Training Completed Successfully on 8GB GPU!")


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/257 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Map:   0%|          | 0/257 [00:00<?, ? examples/s]

Step,Training Loss
25,18.306509
50,5.884922
75,0.322765
100,0.249464
125,0.220129
150,0.200803
175,0.196402
200,0.181484
225,0.179205
250,0.171577


Training Completed Successfully on 8GB GPU!
